# 01 · A polite crawler on a simulated web

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pretraining-data/01_polite_crawler.ipynb) &nbsp; 📖 Concept deck: [How a web crawler works](https://nilesh-salpe.github.io/visual-ai/pretraining-data/02-how-a-crawler-works/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** URL normalisation, a URL-seen set, and a frontier that enforces per-host politeness with a heap of "next allowed time" (the Mercator design), run on a simulated web so no real site is contacted.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import heapq, random
from urllib.parse import urlsplit, urlunsplit

def make_web(n_hosts=20, pages_per_host=30, links=6, seed=0):
    """A simulated web: {url: [outlinks]} plus a per-host download time in seconds. No network access."""
    r = random.Random(seed); hosts = [f"site{h}.example" for h in range(n_hosts)]
    urls = [f"http://{h}/p{i}" for h in hosts for i in range(pages_per_host)]
    web = {}
    for u in urls:
        host = urlsplit(u).netloc
        same = [x for x in urls if urlsplit(x).netloc == host]
        out = r.sample(same, links // 2) + r.sample(urls, links - links // 2)
        # sprinkle in un-normalised variants that should map to the same page
        web[u] = [o if r.random() < 0.7 else o.replace("http://", "HTTP://").replace(".example", ".example:80") + "#frag" for o in out]
    speed = {h: r.uniform(0.05, 0.5) for h in hosts}
    return web, speed

WEB, SPEED = make_web()
SEEDS = ["http://site0.example/p0", "http://site7.example/p3"]

## Task 1: Normalise URLs

Many strings name the same page. Implement `normalize(url)`: lowercase the scheme and host, drop the default port (`:80` for http, `:443` for https), drop the `#fragment`, and use `/` for an empty path.
Without this, a crawler fetches the same page many times.

In [ ]:
def normalize(url):
    # TODO: use urlsplit / urlunsplit
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_normalize(f):
    assert f("HTTP://Site1.Example:80/p3#top") == "http://site1.example/p3"
    assert f("https://a.example:443") == "https://a.example/"
    assert f("http://a.example:8080/x?q=1") == "http://a.example:8080/x?q=1", "keep non-default ports and queries"

check("normalize", _test_normalize, normalize)

## Task 2: A frontier with per-host politeness

Implement a class with `add(url, now)` and `next(now)`:
* URLs are grouped into one FIFO queue **per host**.
* A heap holds `(time_host_may_be_contacted, host)`.
* `next(now)` pops the host with the earliest allowed time; if that time is in the future it returns `(None, wait_until)`; otherwise it returns `(url, host)` from that host's queue. The host goes back on the heap only when `done(host, now, download_time)` is called, with allowed time `now + delay_factor × download_time` (Mercator waited **10×** the last download time).

In [ ]:
class PoliteFrontier:
    def __init__(self, delay_factor=10.0):
        self.delay_factor = delay_factor
        self.queues, self.heap, self.active, self.next_ok = {}, [], set(), {}
    def add(self, url, now=0.0):
        # TODO: append to the host's queue; if the host is idle (not on the heap, not in flight),
        # push (max(now, next_ok.get(host, 0)), host) so an idle host still respects its last delay
        raise NotImplementedError
    def next(self, now):
        # TODO
        raise NotImplementedError
    def done(self, host, now, download_time):
        # TODO: record next_ok[host] = now + delay_factor * download_time; if the host still has URLs, push it back at that time
        raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_PoliteFrontier(cls):
    fr = cls(delay_factor=10.0)
    for u in ["http://a.example/1", "http://a.example/2", "http://b.example/1"]:
        fr.add(u, 0.0)
    u1, h1 = fr.next(0.0); u2, h2 = fr.next(0.0)
    assert {h1, h2} == {"a.example", "b.example"}, "two different hosts can be fetched at once"
    got, wait = fr.next(0.0)
    assert got is None, "a.example is in flight: no third URL until done() is called"
    fr.done("a.example", 1.0, 0.2)
    got, wait = fr.next(1.5)
    assert got is None and abs(wait - 3.0) < 1e-9, "next contact allowed at 1.0 + 10 × 0.2 = 3.0"
    got, host = fr.next(3.0)
    assert got == "http://a.example/2" and host == "a.example"

check("PoliteFrontier", _test_PoliteFrontier, PoliteFrontier)

## Task 3: Run the crawl

Simulate a single-threaded event loop: take the next URL (or jump the clock to `wait_until`), "download" it (advance the clock by the host's `SPEED`), call `done`, normalise its outlinks and add the unseen ones. Stop after `max_pages` downloads.
Return `(pages, elapsed_seconds, violations)`, where a violation is a fetch from a host that starts earlier than `delay_factor ×` that host's previous download time after the previous download ended. A polite crawler has 0. Use your `normalize` and `PoliteFrontier` (or the references).

In [ ]:
def crawl(max_pages=200, delay_factor=10.0, norm=None, Frontier=None):
    norm = norm or normalize; Frontier = Frontier or PoliteFrontier
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_crawl(f):
    pages, t, viol = f(200, 10.0)
    assert len(pages) == 200 and len(set(pages)) == 200, "every page fetched once (normalisation + seen set)"
    assert all("#" not in p and ":80" not in p for p in pages), "outlinks must be normalised"
    assert viol == 0, f"politeness violated {viol} times"

check("crawl", _test_crawl, crawl)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Normalise URLs
def normalize_solution(url):
    p = urlsplit(url); scheme = p.scheme.lower(); host = (p.hostname or "").lower()
    port = p.port
    netloc = host if port is None or (scheme, port) in (("http", 80), ("https", 443)) else f"{host}:{port}"
    return urlunsplit((scheme, netloc, p.path or "/", p.query, ""))

check("normalize (reference)", _test_normalize, normalize_solution)

In [ ]:
# @title Solution for Task 2: A frontier with per-host politeness
class PoliteFrontier_solution:
    def __init__(self, delay_factor=10.0):
        self.delay_factor = delay_factor
        self.queues, self.heap, self.active, self.next_ok = {}, [], set(), {}
    def add(self, url, now=0.0):
        host = urlsplit(url).netloc
        q = self.queues.setdefault(host, [])
        if not q and host not in self.active:
            heapq.heappush(self.heap, (max(now, self.next_ok.get(host, 0.0)), host)); self.active.add(host)
        q.append(url)
    def next(self, now):
        while self.heap:
            t, host = self.heap[0]
            if t > now:
                return None, t
            heapq.heappop(self.heap)
            if self.queues.get(host):
                return self.queues[host].pop(0), host
            self.active.discard(host)
        return None, None
    def done(self, host, now, download_time):
        self.next_ok[host] = now + self.delay_factor * download_time
        if self.queues.get(host):
            heapq.heappush(self.heap, (self.next_ok[host], host))
        else:
            self.active.discard(host)

check("PoliteFrontier (reference)", _test_PoliteFrontier, PoliteFrontier_solution)

In [ ]:
# @title Solution for Task 3: Run the crawl
def crawl_solution(max_pages=200, delay_factor=10.0, norm=None, Frontier=None):
    norm = norm or normalize_solution; Frontier = Frontier or PoliteFrontier_solution
    fr, seen, now, pages, last, viol = Frontier(delay_factor), set(), 0.0, [], {}, 0
    for s in SEEDS:
        s = norm(s); seen.add(s); fr.add(s, now)
    while len(pages) < max_pages:
        url, info = fr.next(now)
        if url is None:
            if info is None: break
            now = info; continue
        host = info
        if host in last and now < last[host][0] + delay_factor * last[host][1] - 1e-9: viol += 1
        now += SPEED[host]; pages.append(url); last[host] = (now, SPEED[host])
        fr.done(host, now, SPEED[host])
        for o in WEB.get(url, []):
            o = norm(o)
            if o not in seen:
                seen.add(o); fr.add(o, now)
    return pages, now, viol

check("crawl (reference)", _test_crawl, crawl_solution)

---
## Experiments
**What a naive FIFO crawler does to servers.** A breadth-first queue with no per-host scheduling fetches whatever comes next, often the same host back to back. Compare the smallest idle gap any host sees between two requests.

In [ ]:
def min_gap(fetches):
    # smallest idle time between the end of one request to a host and the start of the next one to the same host
    last, gap = {}, float("inf")
    for start, h in fetches:
        if h in last: gap = min(gap, start - last[h])
        last[h] = start + SPEED[h]
    return gap
# naive FIFO: no politeness at all
q, seen, now, log = list(SEEDS), set(SEEDS), 0.0, []
while q and len(log) < 300:
    u = q.pop(0); h = urlsplit(u).netloc; log.append((now, h)); now += SPEED[h]
    for o in WEB[u]:
        o = normalize_solution(o)
        if o not in seen: seen.add(o); q.append(o)
print(f"naive FIFO : 300 pages in {now:6.1f} s, smallest gap between requests to one host = {min_gap(log):.2f} s")
fr, seen, now, log = PoliteFrontier_solution(10.0), set(), 0.0, []
for s0 in SEEDS: seen.add(s0); fr.add(s0, 0.0)
while len(log) < 300:
    u, info = fr.next(now)
    if u is None: now = info; continue
    log.append((now, info)); now += SPEED[info]; fr.done(info, now, SPEED[info])
    for o in WEB[u]:
        o = normalize_solution(o)
        if o not in seen: seen.add(o); fr.add(o, now)
print(f"polite (10×): 300 pages in {now:6.1f} s, smallest gap between requests to one host = {min_gap(log):.2f} s")
print("A real crawler hides the waiting by keeping thousands of hosts in flight across many threads.")

---
## Takeaways
* Normalise URLs and keep a seen-set, or the crawl wastes most of its budget on duplicates.
* Politeness is enforced per host: one queue per host and a heap of next-allowed times.
* Mercator waited 10× the last download time before contacting a host again and kept ~3× as many host queues as threads.